# Co-occurrence & Graphe de Synergies — v2 (pondérée)

Améliorations par rapport à v1 :
- **Pondération par placement** : un deck 1er place pèse plus qu'un deck 50e (`1 / placement`)
- **Pondération temporelle** : les decks récents pèsent plus (décroissance exponentielle, demi-vie 365 jours)
- **Quantités réelles** : jouer 3x une carte ≠ 1x (amount normalisé sur 3)
- **Side deck séparé** : co-occurrence side deck → signal sur les menaces anticipées par les joueurs

La table `card_cooccurrence` est reconstruite à partir de ces données qualifiées.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

con = sqlite3.connect('../data/yugioh.db')

# Charger toutes les zones avec placement et date
df = pd.read_sql("""
    SELECT dc.deck_id, dc.card_name, dc.amount, dc.zone,
           td.archetype, td.placement, td.uploaded
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0
""", con)

print(f'Lignes chargées   : {len(df):,}')
print(f'Decks uniques     : {df["deck_id"].nunique():,}')
print(f'Cartes uniques    : {df["card_name"].nunique():,}')
print(f'Zones             : {df["zone"].value_counts().to_dict()}')

Lignes chargées   : 154,541
Decks uniques     : 3,601
Cartes uniques    : 3,243
Zones             : {'main': 79263, 'extra': 47024, 'side': 28254}


## 1. Poids par deck (placement × recency)

In [2]:
REFERENCE_DATE = datetime(2026, 6, 14)
DECAY_DAYS = 365        # demi-vie : un deck vieux d'1 an pèse ~37% d'un deck récent
MAX_PLACEMENT_WEIGHT = 1.0

def placement_weight(p):
    """1er place → 1.0, 2e → 0.5, 10e → 0.1. NULL → 0.3 (poids neutre bas)."""
    if pd.isna(p) or p <= 0:
        return 0.3
    return min(MAX_PLACEMENT_WEIGHT, 1.0 / p)

def recency_weight(uploaded_str):
    """Décroissance exponentielle : exp(-jours / DECAY_DAYS)."""
    if not uploaded_str:
        return 0.5
    try:
        d = datetime.fromisoformat(str(uploaded_str)[:10])
        days_old = max(0, (REFERENCE_DATE - d).days)
        return float(np.exp(-days_old / DECAY_DAYS))
    except Exception:
        return 0.5

# Calculer les poids par deck (une ligne par deck)
deck_meta = (df[['deck_id', 'placement', 'uploaded']]
             .drop_duplicates('deck_id')
             .set_index('deck_id'))

deck_meta['w_placement'] = deck_meta['placement'].apply(placement_weight)
deck_meta['w_recency']   = deck_meta['uploaded'].apply(recency_weight)
deck_meta['weight']      = deck_meta['w_placement'] * deck_meta['w_recency']

# Normaliser : poids moyen = 1 (les scores Jaccard restent comparables à v1)
deck_meta['weight'] = deck_meta['weight'] / deck_meta['weight'].mean()

print('Distribution des poids de decks :')
print(deck_meta['weight'].describe().round(3))
print()
print(f'Deck poids min : {deck_meta["weight"].min():.3f}')
print(f'Deck poids max : {deck_meta["weight"].max():.3f}')

Distribution des poids de decks :
count    3601.000
mean        1.000
std         0.889
min         0.007
25%         0.309
50%         0.763
75%         1.384
max         3.140
Name: weight, dtype: float64

Deck poids min : 0.007
Deck poids max : 3.140


## 2. Matrice pondérée — main deck

In [3]:
# ── Main deck uniquement ──────────────────────────────────────────────────────
main = df[df['zone'] == 'main'].copy()
main = main.merge(deck_meta[['weight']], left_on='deck_id', right_index=True)

# Pondération par quantité : amount normalisé sur 3 (max copies légal)
# card_score(deck, card) = (amount / 3) * deck_weight
main['score'] = (main['amount'].clip(upper=3) / 3.0) * main['weight']

# Matrice : decks × cartes  (score pondéré)
W = main.groupby(['deck_id', 'card_name'])['score'].max().unstack(fill_value=0.0)

# Filtrer : garder seulement les cartes présentes dans au moins 10 decks
card_presence = (W > 0).sum()
W = W[card_presence[card_presence >= 10].index]

print(f'Matrice pondérée main deck : {W.shape[0]} decks × {W.shape[1]} cartes')

Matrice pondérée main deck : 3601 decks × 748 cartes


## 3. Co-occurrence pondérée + Jaccard

In [4]:
# co-occurrence pondérée : W.T @ W
# (W.T @ W)[i,j] = Σ_d  score(d,i) * score(d,j)
m = W.values
cooc_weighted = m.T @ m                      # (nb_cartes × nb_cartes)

# Pour le Jaccard pondéré :
#   count_weighted[i] = Σ_d score(d,i)^2   ← diagonale de cooc_weighted
#   jaccard[i,j] = cooc[i,j] / (count[i] + count[j] - cooc[i,j])
card_counts = np.diag(cooc_weighted)
union = card_counts[:, None] + card_counts[None, :] - cooc_weighted
jaccard = np.where(union > 0, cooc_weighted / union, 0.0)
np.fill_diagonal(jaccard, 0.0)

# Co-occurrence brute (nombre de decks où les deux cartes sont présentes ensemble)
binary = (W.values > 0).astype(np.float32)
cooc_count = (binary.T @ binary).astype(int)

cards = W.columns.tolist()
print(f'Calcul terminé. Shape jaccard : {jaccard.shape}')
print(f'Jaccard max (hors diag) : {np.max(jaccard):.3f}')
print(f'Jaccard moyen (> 0)     : {jaccard[jaccard > 0].mean():.3f}')

Calcul terminé. Shape jaccard : (748, 748)
Jaccard max (hors diag) : 1.000
Jaccard moyen (> 0)     : 0.067


## 4. Top paires

In [5]:
upper = np.triu(jaccard, k=1)
pairs_idx = np.argwhere(upper > 0.1)

pairs = []
for i, j in pairs_idx:
    pairs.append({
        'card_a':     cards[i],
        'card_b':     cards[j],
        'jaccard':    round(float(jaccard[i, j]), 4),
        'cooc_count': int(cooc_count[i, j])
    })

pairs_df = pd.DataFrame(pairs).sort_values('jaccard', ascending=False)
print(f'Paires avec Jaccard > 0.1 : {len(pairs_df):,}')
print()
print('Top 20 paires :')
pairs_df.head(20)

Paires avec Jaccard > 0.1 : 5,489

Top 20 paires :


,card_a,card_b,jaccard,cooc_count
3405,"Hamon, Lord of Striking Thunder - Sacred Beast...",Unleashing the Sacred Beasts,1.0,14
4011,Maliss C MTP-07,Maliss P Dormouse,1.0,136
1354,Couplet the Melodious Songstress,Refrain the Melodious Songstress,1.0,14
5095,Sangen Summoning,Tenpai Dragon Chundra,1.0,14
3402,"Hamon, Lord of Striking Thunder - Sacred Beast...",Skyfire of the Sacred Beast,1.0,14
3401,"Hamon, Lord of Striking Thunder - Sacred Beast...","Raviel, Lord of Phantasms - Sacred Beast of En...",1.0,14
3400,"Hamon, Lord of Striking Thunder - Sacred Beast...",Martyr of the Sacred Beasts,1.0,14
5388,The Monarchs Masterplan,The Monarchs Revolt,1.0,13
3178,Gem-Knight Dispersion,Gem-Knight Nepyrim,1.0,13
3070,Funny Dark Rabbit,Toon World the Perfect World,1.0,46


## 5. Co-occurrence par archetype

In [6]:
def top_pairs_for_archetype(archetype, min_jaccard=0.5, top_n=15):
    """Co-occurrence pondérée sur les decks d'un archetype donné."""
    deck_ids = df[(df['zone'] == 'main') & (df['archetype'] == archetype)]['deck_id'].unique()
    sub_W = W.loc[W.index.isin(deck_ids)]
    sub_W = sub_W.loc[:, (sub_W > 0).sum() > 0]

    if sub_W.shape[0] < 5:
        print(f'Pas assez de decks pour {archetype} ({sub_W.shape[0]})')
        return

    m2 = sub_W.values
    c2 = m2.T @ m2
    d2 = np.diag(c2)
    u2 = d2[:, None] + d2[None, :] - c2
    j2 = np.where(u2 > 0, c2 / u2, 0.0)
    np.fill_diagonal(j2, 0.0)

    bin2 = (sub_W.values > 0).astype(np.float32)
    cnt2 = (bin2.T @ bin2).astype(int)

    local_cards = sub_W.columns.tolist()
    idx2 = np.argwhere(np.triu(j2, k=1) >= min_jaccard)
    result = [{'card_a': local_cards[i], 'card_b': local_cards[j],
               'jaccard': round(float(j2[i,j]), 3), 'count': int(cnt2[i,j])}
              for i, j in idx2]

    return pd.DataFrame(result).sort_values('jaccard', ascending=False).head(top_n)

print('=== Maliss ===')
display(top_pairs_for_archetype('Maliss'))
print('=== Tenpai Dragon ===')
display(top_pairs_for_archetype('Tenpai Dragon'))
print('=== Snake-Eye ===')
display(top_pairs_for_archetype('Snake-Eye'))

=== Maliss ===


,card_a,card_b,jaccard,count
171,"World Legacy - ""World Crown""","World Legacy - ""World Wand""",1.0,2
109,Magician of Dark Chaos - Black Chaos,Skull Archfiend of Chaos,1.0,1
149,Mystical Space Typhoon,Radiant Typhoon Swen,1.0,1
148,Mystical Space Typhoon,Radiant Typhoon Eldam,1.0,1
147,Mind Shuffle,Skull Archfiend of Chaos,1.0,1
146,Mind Shuffle,Ritual of Light and Darkness,1.0,1
138,Maliss P Dormouse,Wizard @Ignister,1.0,136
126,Maliss C MTP-07,Wizard @Ignister,1.0,136
120,Maliss C MTP-07,Maliss P Dormouse,1.0,136
112,Magicians' Souls,Skull Archfiend of Chaos,1.0,1


=== Tenpai Dragon ===


,card_a,card_b,jaccard,count
249,The Gaze of Timaeus,The Hallowed Azamina,1.0,1
32,Bystial Baldrake,Ghost Ogre & Snow Rabbit,1.0,1
43,Bystial Druiswurm,Ghost Ogre & Snow Rabbit,1.0,1
195,PSY-Frame Driver,PSY-Framegear Gamma,1.0,1
169,K9-17 Izuna,Super Polymerization,1.0,1
164,"Incredible Ecclesia, the Virtuous",WANTED: Seeker of Sinful Spoils,1.0,1
158,Gold Sarcophagus,PSY-Framegear Gamma,1.0,1
157,Gold Sarcophagus,PSY-Frame Driver,1.0,1
149,Gandora-G the Dragon of Destruction,The Hallowed Azamina,1.0,1
148,Gandora-G the Dragon of Destruction,The Gaze of Timaeus,1.0,1


=== Snake-Eye ===
Pas assez de decks pour Snake-Eye (1)


None

## 6. Staples universelles

In [7]:
# Fréquence pondérée par deck : sum des scores / sum des poids de tous les decks
total_weight = deck_meta.loc[W.index, 'weight'].sum()
card_freq_weighted = W.sum(axis=0) / total_weight

staples = card_freq_weighted[card_freq_weighted > 0.2].sort_values(ascending=False)
print(f'Cartes présentes dans >20% des decks (pondéré) : {len(staples)}')
for card, freq in staples.items():
    print(f'  {freq:.0%}  {card}')

Cartes présentes dans >20% des decks (pondéré) : 10
  66%  Ash Blossom & Joyous Spring
  63%  Mulcharmy Fuwalos
  35%  Ghost Belle & Haunted Mansion
  34%  Infinite Impermanence
  27%  The Fallen & The Virtuous
  25%  Effect Veiler
  25%  Fydraulis Harmonia
  21%  Forbidden Droplet
  20%  Called by the Grave
  20%  Droll & Lock Bird


## 7. Side deck — menaces anticipées

In [8]:
# Le side deck révèle quelles cartes les joueurs mettent pour contrer la méta
side = df[df['zone'] == 'side'].copy()
side = side.merge(deck_meta[['weight']], left_on='deck_id', right_index=True)
side['score'] = (side['amount'].clip(upper=3) / 3.0) * side['weight']

W_side = side.groupby(['deck_id', 'card_name'])['score'].max().unstack(fill_value=0.0)

# Filtrer : au moins 5 decks
card_presence_side = (W_side > 0).sum()
W_side = W_side[card_presence_side[card_presence_side >= 5].index]

print(f'Matrice side deck : {W_side.shape[0]} decks × {W_side.shape[1]} cartes')
print()

# Co-occurrence side
ms = W_side.values
cooc_side = ms.T @ ms
cs = np.diag(cooc_side)
us = cs[:, None] + cs[None, :] - cooc_side
jaccard_side = np.where(us > 0, cooc_side / us, 0.0)
np.fill_diagonal(jaccard_side, 0.0)

bin_s = (W_side.values > 0).astype(np.float32)
count_side = (bin_s.T @ bin_s).astype(int)

cards_side = W_side.columns.tolist()

# Cartes les plus présentes en side
total_weight_side = deck_meta.loc[deck_meta.index.isin(W_side.index), 'weight'].sum()
side_freq = W_side.sum(axis=0) / total_weight_side
top_side = side_freq.sort_values(ascending=False).head(30)

print('Top 30 cartes en side deck (pondéré) :')
for card, freq in top_side.items():
    print(f'  {freq:.0%}  {card}')

Matrice side deck : 3319 decks × 260 cartes

Top 30 cartes en side deck (pondéré) :
  52%  Mulcharmy Purulia
  30%  Solemn Judgment
  24%  Mulcharmy Fuwalos
  24%  Droll & Lock Bird
  22%  Harpie's Feather Duster
  20%  Solemn Warning
  14%  Retaliating "C"
  13%  Solemn Accusation
  13%  Nibiru, the Primal Being
  13%  Lightning Storm
  12%  Ghost Ogre & Snow Rabbit
  12%  Dimensional Barrier
  11%  Ghost Belle & Haunted Mansion
  10%  Evenly Matched
  10%  Heavy Storm
  8%  Triple Tactics Thrust
  8%  Dimensional Fissure
  7%  Called by the Grave
  7%  Red Reboot
  7%  Infinite Impermanence
  7%  Artifact Lancea
  5%  Lava Golem
  5%  Triple Tactics Talent
  5%  PSY-Framegear Delta
  5%  Forbidden Droplet
  5%  Mistaken Arrest
  5%  Pot of Sloth
  5%  Ash Blossom & Joyous Spring
  4%  Dark Ruler No More
  4%  Ultimate Slayer


## 8. Sauvegarder en base

In [9]:
con2 = sqlite3.connect('../data/yugioh.db')

# ── Main deck co-occurrence ────────────────────────────────────────────────────
significant = pairs_df[pairs_df['jaccard'] > 0.05].copy()

con2.execute("DROP TABLE IF EXISTS card_cooccurrence")
con2.execute("""
    CREATE TABLE card_cooccurrence (
        card_a      TEXT,
        card_b      TEXT,
        jaccard     REAL,
        cooc_count  INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
significant.to_sql('card_cooccurrence', con2, if_exists='append', index=False)

# ── Side deck co-occurrence ────────────────────────────────────────────────────
upper_s = np.triu(jaccard_side, k=1)
pairs_side_idx = np.argwhere(upper_s > 0.05)
pairs_side = [{'card_a': cards_side[i], 'card_b': cards_side[j],
               'jaccard': round(float(jaccard_side[i,j]), 4),
               'cooc_count': int(count_side[i,j])}
              for i, j in pairs_side_idx]
pairs_side_df = pd.DataFrame(pairs_side)

con2.execute("DROP TABLE IF EXISTS card_cooccurrence_side")
con2.execute("""
    CREATE TABLE card_cooccurrence_side (
        card_a      TEXT,
        card_b      TEXT,
        jaccard     REAL,
        cooc_count  INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
if not pairs_side_df.empty:
    pairs_side_df.to_sql('card_cooccurrence_side', con2, if_exists='append', index=False)

con2.commit()
con2.close()

print(f'✓ {len(significant):,} paires main deck sauvegardées dans card_cooccurrence')
print(f'✓ {len(pairs_side_df):,} paires side deck sauvegardées dans card_cooccurrence_side')

✓ 5,489 paires main deck sauvegardées dans card_cooccurrence
✓ 739 paires side deck sauvegardées dans card_cooccurrence_side


## 9. Co-occurrence 90 jours — fenêtre glissante récente (TOK-11)

Signal court-terme : uniquement les decks des 90 derniers jours.
→ Table `card_cooccurrence_90d` (même schéma que `card_cooccurrence`)

In [10]:
from datetime import timedelta

CUTOFF_90D = REFERENCE_DATE - timedelta(days=90)
cutoff_str = CUTOFF_90D.strftime('%Y-%m-%d')

# Filtrer df sur les 90 derniers jours
df_90d = df[df['uploaded'] >= cutoff_str].copy()
deck_ids_90d = df_90d['deck_id'].unique()

print(f'Filtre 90j : >= {cutoff_str}')
print(f'Decks 90j  : {len(deck_ids_90d):,} (vs {df["deck_id"].nunique():,} total)')
print(f'Lignes 90j : {len(df_90d):,}')

# Poids pour les decks 90j (placement uniquement — recency déjà filtrée par la fenêtre)
deck_meta_90d = (df_90d[['deck_id', 'placement', 'uploaded']]
                 .drop_duplicates('deck_id')
                 .set_index('deck_id'))
deck_meta_90d['w_placement'] = deck_meta_90d['placement'].apply(placement_weight)
deck_meta_90d['w_recency']   = deck_meta_90d['uploaded'].apply(recency_weight)
deck_meta_90d['weight']      = deck_meta_90d['w_placement'] * deck_meta_90d['w_recency']
deck_meta_90d['weight']      = deck_meta_90d['weight'] / deck_meta_90d['weight'].mean()

# Main deck 90j
main_90d = df_90d[df_90d['zone'] == 'main'].copy()
main_90d = main_90d.merge(deck_meta_90d[['weight']], left_on='deck_id', right_index=True)
main_90d['score'] = (main_90d['amount'].clip(upper=3) / 3.0) * main_90d['weight']

W_90d = main_90d.groupby(['deck_id', 'card_name'])['score'].max().unstack(fill_value=0.0)

# Min 5 decks (seuil réduit car fenêtre plus courte)
card_presence_90d = (W_90d > 0).sum()
W_90d = W_90d[card_presence_90d[card_presence_90d >= 5].index]

print(f'\nMatrice 90j : {W_90d.shape[0]} decks × {W_90d.shape[1]} cartes')

# Jaccard pondéré
m_90d = W_90d.values
cooc_90d = m_90d.T @ m_90d
counts_90d = np.diag(cooc_90d)
union_90d  = counts_90d[:, None] + counts_90d[None, :] - cooc_90d
jaccard_90d = np.where(union_90d > 0, cooc_90d / union_90d, 0.0)
np.fill_diagonal(jaccard_90d, 0.0)

bin_90d  = (W_90d.values > 0).astype(np.float32)
cooc_cnt_90d = (bin_90d.T @ bin_90d).astype(int)
cards_90d = W_90d.columns.tolist()

upper_90d = np.triu(jaccard_90d, k=1)
pairs_90d = [{'card_a': cards_90d[i], 'card_b': cards_90d[j],
               'jaccard': round(float(jaccard_90d[i,j]), 4),
               'cooc_count': int(cooc_cnt_90d[i,j])}
             for i, j in np.argwhere(upper_90d > 0.05)]
pairs_90d_df = pd.DataFrame(pairs_90d).sort_values('jaccard', ascending=False)

print(f'Paires Jaccard > 0.05 : {len(pairs_90d_df):,}')
print('\nTop 10 paires 90j :')
print(pairs_90d_df.head(10).to_string(index=False))

# Sauvegarder
con3 = sqlite3.connect('../data/yugioh.db')
con3.execute('DROP TABLE IF EXISTS card_cooccurrence_90d')
con3.execute("""
    CREATE TABLE card_cooccurrence_90d (
        card_a      TEXT,
        card_b      TEXT,
        jaccard     REAL,
        cooc_count  INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
pairs_90d_df.to_sql('card_cooccurrence_90d', con3, if_exists='append', index=False)
con3.commit()
con3.close()
print(f'\n✓ {len(pairs_90d_df):,} paires sauvegardées dans card_cooccurrence_90d')

Filtre 90j : >= 2026-03-16
Decks 90j  : 2,352 (vs 3,601 total)
Lignes 90j : 102,512

Matrice 90j : 2352 decks × 758 cartes
Paires Jaccard > 0.05 : 8,797

Top 10 paires 90j :
                                  card_a                       card_b  jaccard  cooc_count
                      Atlantean Dragoons         Mermail Shadow Squad      1.0          16
Mementoal Tecuhtlica - Combined Creation         Mementotlan Akihiron      1.0          12
             Martyr of the Sacred Beasts  Skyfire of the Sacred Beast      1.0          14
                  Enneacraft - Atori.MAR   Proto Enneacraft - "orgIA"      1.0          11
             Martyr of the Sacred Beasts Unleashing the Sacred Beasts      1.0          14
                             Mask Change               Miracle Fusion      1.0           6
              Elemental HERO Shadow Mist             Favorite Contact      1.0           6
                     Elemental HERO Neos               Miracle Fusion      1.0           6
       

## 10. Co-occurrence Extra Deck (TOK-14)

L'extra deck révèle les **win conditions et engines** des decks.
Analyse séparée du main deck :
- Amount normalisé sur 1 (la plupart des extra deck cards sont en 1 exemplaire)
- Seuil min 5 decks (extra deck plus homogène → signal plus fort)
- Tables : `card_cooccurrence_extra` + `archetype_extra_profile`

In [11]:
# ── Extra deck uniquement ──────────────────────────────────────────────────────
extra = df[df['zone'] == 'extra'].copy()
extra = extra.merge(deck_meta[['weight']], left_on='deck_id', right_index=True)

# Normalisation sur 1 (extra deck cards généralement en 1 exemplaire max)
# On garde amount/1 clippé à 1 pour traiter 2+ copies comme 1
extra['score'] = (extra['amount'].clip(upper=1) / 1.0) * extra['weight']

W_ex = extra.groupby(['deck_id', 'card_name'])['score'].max().unstack(fill_value=0.0)

# Seuil : au moins 5 decks
card_pres_ex = (W_ex > 0).sum()
W_ex = W_ex[card_pres_ex[card_pres_ex >= 5].index]

print(f'Matrice extra deck : {W_ex.shape[0]} decks × {W_ex.shape[1]} cartes')

# Jaccard pondéré extra deck
m_ex    = W_ex.values
cooc_ex = m_ex.T @ m_ex
diag_ex = np.diag(cooc_ex)
union_ex = diag_ex[:, None] + diag_ex[None, :] - cooc_ex
jaccard_ex = np.where(union_ex > 0, cooc_ex / union_ex, 0.0)
np.fill_diagonal(jaccard_ex, 0.0)

bin_ex   = (W_ex.values > 0).astype(np.float32)
count_ex = (bin_ex.T @ bin_ex).astype(int)
cards_ex = W_ex.columns.tolist()

upper_ex   = np.triu(jaccard_ex, k=1)
pairs_ex   = [{'card_a': cards_ex[i], 'card_b': cards_ex[j],
                'jaccard': round(float(jaccard_ex[i,j]), 4),
                'cooc_count': int(count_ex[i,j])}
               for i, j in np.argwhere(upper_ex > 0.05)]
pairs_ex_df = pd.DataFrame(pairs_ex).sort_values('jaccard', ascending=False)

print(f'Paires Jaccard > 0.05 : {len(pairs_ex_df):,}')
print('\nTop 15 paires extra deck :')
print(pairs_ex_df.head(15).to_string(index=False))

Matrice extra deck : 3598 decks × 616 cartes
Paires Jaccard > 0.05 : 5,583

Top 15 paires extra deck :
                          card_a                                     card_b  jaccard  cooc_count
           Gem-Knight Aquamarine                           Gem-Knight Topaz      1.0          12
     Gimmick Puppet Chimera Doll    Number 15: Gimmick Puppet Giant Grinder      1.0           5
          Stellarknight Delteros                    Tellarknight Ptolemaeus      1.0           5
Gimmick Puppet Fantasix Machinix               Gimmick Puppet Gigantes Doll      1.0           5
      D/D/D Abyss King Gilgamesh                   D/D/D Flame King Genghis      1.0          14
      D/D/D Abyss King Gilgamesh                   D/D/D Marksman King Tell      1.0          14
      D/D/D Abyss King Gilgamesh               D/D/D Sky King Zeus Ragnarok      1.0          14
     Gimmick Puppet Chimera Doll Number C40: Gimmick Puppet of Dark Strings      1.0           5
      D/D/D Abyss King G

In [12]:
# ── Profil extra deck par archetype ───────────────────────────────────────────
# extra hérite déjà de la colonne archetype via df (JOIN tournament_decks)
arch_extra = extra.dropna(subset=['archetype']).copy()

# Poids total par archetype
arch_weights = arch_extra.drop_duplicates('deck_id').groupby('archetype')['weight'].sum().rename('total_weight')

# Score agrégé par (archetype, card_name)
arch_card_score = (arch_extra
                   .groupby(['archetype', 'card_name'])['score']
                   .sum()
                   .reset_index())

arch_card_score = arch_card_score.merge(arch_weights, on='archetype')

# Fréquence pondérée = score / total_weight
arch_card_score['freq'] = arch_card_score['score'] / arch_card_score['total_weight']

# Nombre de decks brut par (archetype, card)
deck_count_per_arch = (arch_extra
                       .groupby(['archetype', 'card_name'])['deck_id']
                       .nunique()
                       .reset_index()
                       .rename(columns={'deck_id': 'n_decks'}))

arch_card_score = arch_card_score.merge(deck_count_per_arch, on=['archetype', 'card_name'])

# Garder uniquement freq > 0.1 (carte utilisée dans au moins 10% des decks de l'archetype)
profile = arch_card_score[arch_card_score['freq'] >= 0.10].copy()

# Identifier les cartes "génériques" (utilisées dans >= 10 archetypes différents)
card_arch_count = profile.groupby('card_name')['archetype'].nunique().rename('n_archetypes')
profile = profile.merge(card_arch_count.reset_index(), on='card_name')
profile['is_generic'] = profile['n_archetypes'] >= 10

print(f'Profil extra deck : {len(profile):,} lignes (archetype × carte, freq >= 10%)')
print(f'Archetypes avec profil : {profile["archetype"].nunique()}')
print(f'Cartes génériques (>=10 archetypes) : {profile[profile["is_generic"]]["card_name"].nunique()} cartes')

print('\n=== Top extra deck génériques ===')
generics = (profile[profile['is_generic']]
            .groupby('card_name')
            .agg(n_archetypes=('archetype', 'nunique'), avg_freq=('freq', 'mean'))
            .sort_values('n_archetypes', ascending=False)
            .head(20))
print(generics.to_string(float_format='{:.2f}'.format))

print('\n=== Extra deck top archetype ===')
top_arch = profile['archetype'].value_counts().index[0]
print(f'Archetype le plus représenté : {top_arch}')
kewl = profile[profile['archetype'] == top_arch].sort_values('freq', ascending=False)
print(kewl[['card_name', 'freq', 'n_decks', 'is_generic']].head(15).to_string(index=False, float_format='{:.2f}'.format))

Profil extra deck : 2,961 lignes (archetype × carte, freq >= 10%)
Archetypes avec profil : 146
Cartes génériques (>=10 archetypes) : 62 cartes

=== Top extra deck génériques ===
                                              n_archetypes  avg_freq
card_name                                                           
S:P Little Knight                                      100      0.79
Super Starslayer TY-PHON - Sky Crisis                   55      0.60
I:P Masquerena                                          46      0.66
Garura, Wings of Resonant Life                          45      0.58
Mudragon of the Swamp                                   38      0.50
Divine Arsenal AA-ZEUS - Sky Thunder                    36      0.74
Albion the Branded Dragon                               31      0.44
Ecclesia and the Dark Dragon                            28      0.45
Accesscode Talker                                       23      0.56
Evilswarm Exciton Knight                                23     

In [13]:
# ── Sauvegarder en base ────────────────────────────────────────────────────────
con_ex = sqlite3.connect('../data/yugioh.db')

# card_cooccurrence_extra
con_ex.execute("DROP TABLE IF EXISTS card_cooccurrence_extra")
con_ex.execute("""
    CREATE TABLE card_cooccurrence_extra (
        card_a      TEXT,
        card_b      TEXT,
        jaccard     REAL,
        cooc_count  INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
pairs_ex_df.to_sql('card_cooccurrence_extra', con_ex, if_exists='append', index=False)

# archetype_extra_profile
con_ex.execute("DROP TABLE IF EXISTS archetype_extra_profile")
con_ex.execute("""
    CREATE TABLE archetype_extra_profile (
        archetype    TEXT,
        card_name    TEXT,
        freq         REAL,
        n_decks      INTEGER,
        n_archetypes INTEGER,
        is_generic   INTEGER,
        PRIMARY KEY (archetype, card_name)
    )
""")
save_profile = profile[['archetype','card_name','freq','n_decks','n_archetypes','is_generic']].copy()
save_profile['is_generic'] = save_profile['is_generic'].astype(int)
save_profile.to_sql('archetype_extra_profile', con_ex, if_exists='append', index=False)

con_ex.commit()
con_ex.close()

print(f'✓ card_cooccurrence_extra    : {len(pairs_ex_df):,} paires')
print(f'✓ archetype_extra_profile    : {len(save_profile):,} lignes ({save_profile["archetype"].nunique()} archetypes)')

✓ card_cooccurrence_extra    : 5,583 paires
✓ archetype_extra_profile    : 2,961 lignes (146 archetypes)


## 11. Co-occurrence Elite (TOK-19) — YCS / Nationals / WCQ uniquement

## 11. Co-occurrence Elite — YCS / Nationals / WCQ (TOK-19)

In [14]:
ELITE_TYPES = (
    "Yu-Gi-Oh! Championship Series",
    "Yu-Gi-Oh! National Championship",
    "Yu-Gi-Oh! World Champ Qualifier",
    "Premiere Event",
    "Yu-Gi-Oh! World Championship Finals",
)

df_elite = pd.read_sql(f"""
    SELECT dc.deck_id, dc.card_name, dc.amount, dc.zone,
           td.archetype, td.placement, td.uploaded
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0
      AND td.tournament_type IN ({','.join(["'"+t+"'" for t in ELITE_TYPES])})
""", con)

print(f"Elite decks : {df_elite['deck_id'].nunique():,} (vs {df['deck_id'].nunique():,} total)")

# Poids par deck elite
deck_meta_elite = (df_elite[['deck_id','placement','uploaded']]
                   .drop_duplicates('deck_id').set_index('deck_id'))
deck_meta_elite['w_placement'] = deck_meta_elite['placement'].apply(placement_weight)
deck_meta_elite['w_recency']   = deck_meta_elite['uploaded'].apply(recency_weight)
deck_meta_elite['weight']      = deck_meta_elite['w_placement'] * deck_meta_elite['w_recency']
deck_meta_elite['weight']      = deck_meta_elite['weight'] / deck_meta_elite['weight'].mean()

main_el = df_elite[df_elite['zone'] == 'main'].copy()
main_el = main_el.merge(deck_meta_elite[['weight']], left_on='deck_id', right_index=True)
main_el['score'] = (main_el['amount'].clip(upper=3) / 3.0) * main_el['weight']

W_el = main_el.groupby(['deck_id','card_name'])['score'].max().unstack(fill_value=0.0)
card_pres_el = (W_el > 0).sum()
W_el = W_el[card_pres_el[card_pres_el >= 5].index]

m_el = W_el.values
cooc_el = m_el.T @ m_el
diag_el = np.diag(cooc_el)
union_el = diag_el[:,None] + diag_el[None,:] - cooc_el
jac_el = np.where(union_el > 0, cooc_el / union_el, 0.0)
np.fill_diagonal(jac_el, 0.0)
bin_el = (W_el.values > 0).astype(np.float32)
cnt_el = (bin_el.T @ bin_el).astype(int)
cards_el = W_el.columns.tolist()

upper_el = np.triu(jac_el, k=1)
pairs_el = [{'card_a': cards_el[i], 'card_b': cards_el[j],
              'jaccard': round(float(jac_el[i,j]),4), 'cooc_count': int(cnt_el[i,j])}
            for i,j in np.argwhere(upper_el > 0.05)]
pairs_el_df = pd.DataFrame(pairs_el).sort_values('jaccard', ascending=False)

print(f"Paires Jaccard > 0.05 elite : {len(pairs_el_df):,}")
print("\nTop 10 paires elite :")
print(pairs_el_df.head(10).to_string(index=False))

con_el = sqlite3.connect('../data/yugioh.db')
con_el.execute("DROP TABLE IF EXISTS card_cooccurrence_elite")
con_el.execute("""
    CREATE TABLE card_cooccurrence_elite (
        card_a TEXT, card_b TEXT, jaccard REAL, cooc_count INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
pairs_el_df.to_sql('card_cooccurrence_elite', con_el, if_exists='append', index=False)
con_el.commit(); con_el.close()
print(f"\n✓ {len(pairs_el_df):,} paires dans card_cooccurrence_elite")

Elite decks : 776 (vs 3,601 total)
Paires Jaccard > 0.05 elite : 4,796

Top 10 paires elite :
                 card_a                        card_b  jaccard  cooc_count
     Runick Destruction        Runick Freezing Curses      1.0           9
Fire King Avatar Arvata Fire King High Avatar Garunix      1.0           6
         Mementomictlan          Mementotlan Akihiron      1.0          14
       PSY-Frame Driver           PSY-Framegear Gamma      1.0          48
  Labrynth Chandraglier        Labrynth Stovie Torbie      1.0           9
         Mementomictlan              Mementotlan Mace      1.0          14
        Despian Tragedy            Fusion Duplication      1.0           5
 Once Upon a Fairy Tail      Tails of the Fairy Tails      1.0          11
   Mementotlan Akihiron              Mementotlan Mace      1.0          14
          Kewl Tune Cue              Kewl Tune Rotary      1.0         112

✓ 4,796 paires dans card_cooccurrence_elite


## 12. Co-occurrence par trimestre (TOK-18)

In [15]:
# Co-occurrence par trimestre (TOK-18)
# Détecte l'apparition/disparition de synergies au fil des saisons

def get_quarter(uploaded_str):
    if not uploaded_str: return None
    d = str(uploaded_str)[:7]  # YYYY-MM
    y, m = int(d[:4]), int(d[5:7])
    q = (m - 1) // 3 + 1
    return f"{y}-Q{q}"

df['quarter'] = df['uploaded'].apply(get_quarter)
quarters = sorted(df['quarter'].dropna().unique())
print(f"Trimestres disponibles : {quarters}")

quarterly_rows = []
for qtr in quarters:
    df_q = df[df['quarter'] == qtr]
    n_decks_q = df_q['deck_id'].nunique()
    if n_decks_q < 20:
        print(f"  {qtr}: {n_decks_q} decks — skip (< 20)")
        continue

    dm_q = (df_q[['deck_id','placement','uploaded']]
            .drop_duplicates('deck_id').set_index('deck_id'))
    dm_q['w_placement'] = dm_q['placement'].apply(placement_weight)
    dm_q['w_recency']   = dm_q['uploaded'].apply(recency_weight)
    dm_q['weight'] = dm_q['w_placement'] * dm_q['w_recency']
    dm_q['weight'] = dm_q['weight'] / dm_q['weight'].mean()

    main_q = df_q[df_q['zone'] == 'main'].merge(dm_q[['weight']], left_on='deck_id', right_index=True)
    main_q['score'] = (main_q['amount'].clip(upper=3) / 3.0) * main_q['weight']

    W_q = main_q.groupby(['deck_id','card_name'])['score'].max().unstack(fill_value=0.0)
    W_q = W_q.loc[:, (W_q > 0).sum() >= 5]
    if W_q.shape[1] < 2: continue

    m_q = W_q.values
    c_q = m_q.T @ m_q
    d_q = np.diag(c_q)
    u_q = d_q[:,None] + d_q[None,:] - c_q
    j_q = np.where(u_q > 0, c_q / u_q, 0.0)
    np.fill_diagonal(j_q, 0.0)
    cards_q = W_q.columns.tolist()

    top_pairs = np.argwhere(np.triu(j_q, k=1) > 0.2)
    for i, j in top_pairs:
        quarterly_rows.append({
            'quarter': qtr, 'card_a': cards_q[i], 'card_b': cards_q[j],
            'jaccard': round(float(j_q[i,j]), 4)
        })
    print(f"  {qtr}: {n_decks_q} decks, {len(top_pairs)} paires > 0.2")

qtr_df = pd.DataFrame(quarterly_rows)
print(f"\nTotal : {len(qtr_df):,} lignes (quarter × paire)")

con_q = sqlite3.connect('../data/yugioh.db')
con_q.execute("DROP TABLE IF EXISTS card_cooccurrence_quarterly")
con_q.execute("""
    CREATE TABLE card_cooccurrence_quarterly (
        quarter TEXT, card_a TEXT, card_b TEXT, jaccard REAL,
        PRIMARY KEY (quarter, card_a, card_b)
    )
""")
qtr_df.to_sql('card_cooccurrence_quarterly', con_q, if_exists='append', index=False)
con_q.commit(); con_q.close()
print(f"✓ {len(qtr_df):,} lignes dans card_cooccurrence_quarterly")

Trimestres disponibles : ['2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4', '2026-Q1', '2026-Q2']
  2024-Q1: 35 decks, 175 paires > 0.2
  2024-Q2: 48 decks, 430 paires > 0.2
  2024-Q3: 17 decks — skip (< 20)
  2024-Q4: 73 decks, 536 paires > 0.2
  2025-Q1: 56 decks, 420 paires > 0.2
  2025-Q2: 103 decks, 755 paires > 0.2
  2025-Q3: 63 decks, 460 paires > 0.2
  2025-Q4: 109 decks, 774 paires > 0.2
  2026-Q1: 1070 decks, 2580 paires > 0.2
  2026-Q2: 2027 decks, 3721 paires > 0.2

Total : 9,851 lignes (quarter × paire)


✓ 9,851 lignes dans card_cooccurrence_quarterly


## 13. Segmentation OCG vs TCG (TOK-20)

In [16]:
# Segmentation OCG vs TCG (TOK-20)
# ocg=1 → tournois asiatiques (Japon/Corée/Asie)
# ocg=0 → tournois TCG (NA/EU/Océanie)
# tournament_location est vide — on utilise le flag ocg comme proxy régional

df_reg = pd.read_sql("""
    SELECT dc.deck_id, dc.card_name, dc.amount, dc.zone,
           td.archetype, td.placement, td.uploaded, td.ocg,
           substr(td.uploaded,1,7) AS month
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0 AND dc.zone = 'main'
      AND td.archetype IS NOT NULL
""", con)

for region, ocg_val in [('TCG', 0), ('OCG', 1)]:
    sub = df_reg[df_reg['ocg'] == ocg_val]
    n_decks = sub['deck_id'].nunique()
    n_months = sub['month'].nunique()
    print(f"{region} (ocg={ocg_val}): {n_decks:,} decks, {n_months} mois")

# Meta score régional : share & avg_placement par (region, month, archetype)
regional_rows = []
for region, ocg_val in [('TCG', 0), ('OCG', 1)]:
    sub = df_reg[df_reg['ocg'] == ocg_val].drop_duplicates(['deck_id','archetype'])
    
    for month, grp in sub.groupby('month'):
        total = len(grp)
        if total < 5: continue
        arch_counts = grp.groupby('archetype').agg(
            n_decks=('deck_id','nunique'),
            avg_placement=('placement','mean')
        ).reset_index()
        arch_counts['share'] = arch_counts['n_decks'] / total
        arch_counts['meta_score'] = arch_counts['share'] * 100 * (
            1 / (arch_counts['avg_placement'].fillna(50) + 1)
        ) * 10
        arch_counts['month']  = month
        arch_counts['region'] = region
        regional_rows.append(arch_counts)

regional_df = pd.concat(regional_rows, ignore_index=True)
print(f"\nMeta régional : {len(regional_df):,} lignes ({regional_df['archetype'].nunique()} archetypes)")

con_r = sqlite3.connect('../data/yugioh.db')
con_r.execute("DROP TABLE IF EXISTS meta_scores_regional")
con_r.execute("""
    CREATE TABLE meta_scores_regional (
        region TEXT, month TEXT, archetype TEXT,
        n_decks INTEGER, share REAL, avg_placement REAL, meta_score REAL,
        PRIMARY KEY (region, month, archetype)
    )
""")
regional_df[['region','month','archetype','n_decks','share','avg_placement','meta_score']].to_sql(
    'meta_scores_regional', con_r, if_exists='append', index=False
)
con_r.commit(); con_r.close()
print(f"✓ {len(regional_df):,} lignes dans meta_scores_regional")

# Diff OCG vs TCG : archetypes plus forts en OCG qu'en TCG
print("\n=== OCG vs TCG — écarts de share (mois récents 2026) ===")
recent = regional_df[regional_df['month'] >= '2026-01']
pivot_reg = recent.groupby(['archetype','region'])['share'].mean().unstack(fill_value=0)
if 'OCG' in pivot_reg and 'TCG' in pivot_reg:
    pivot_reg['ocg_advantage'] = pivot_reg['OCG'] - pivot_reg['TCG']
    top_ocg = pivot_reg.sort_values('ocg_advantage', ascending=False).head(10)
    top_tcg = pivot_reg.sort_values('ocg_advantage').head(10)
    print("Top OCG (plus fort en OCG qu'en TCG) :")
    print(top_ocg[['OCG','TCG','ocg_advantage']].to_string(float_format='{:.3f}'.format))
    print("\nTop TCG (plus fort en TCG qu'en OCG) :")
    print(top_tcg[['OCG','TCG','ocg_advantage']].to_string(float_format='{:.3f}'.format))

TCG (ocg=0): 2,008 decks, 30 mois
OCG (ocg=1): 1,593 decks, 15 mois

Meta régional : 678 lignes (145 archetypes)
✓ 678 lignes dans meta_scores_regional

=== OCG vs TCG — écarts de share (mois récents 2026) ===
Top OCG (plus fort en OCG qu'en TCG) :
region             OCG   TCG  ocg_advantage
archetype                                  
Toon             0.170 0.000          0.170
Kewl Tune        0.216 0.083          0.133
Chaos Ritual     0.091 0.000          0.091
Sky Striker      0.101 0.031          0.070
Elfnote          0.084 0.029          0.055
Ryzeal Mitsurugi 0.032 0.008          0.024
Blitzclique      0.024 0.000          0.024
Witchcrafter     0.022 0.000          0.022
Purrely          0.021 0.000          0.021
Goblin Biker     0.017 0.000          0.017

Top TCG (plus fort en TCG qu'en OCG) :
region                  OCG   TCG  ocg_advantage
archetype                                       
Dracotail             0.009 0.133         -0.124
Azamina               0.000 0.069   